# Apartment buildings in central Copenhagen: EDA and RAG material for the capacity estimate

**Goal:** build RAG material for the LLM API so it can estimate the **capacity of an apartment building** from a request.
The scope is apartment buildings (OSM `apartments` and `residential`): they are about 93% of the data.
Houses, terraced buildings and dormitories are left out.

**What capacity means here:** the *typical* number of residents when the building is normally occupied, not the legal
maximum. The legal limit (20 m2 per person) is only used as a ceiling.

**What the data has:** footprint 100%, levels about 65%, roof levels about 53%, height about 1%, flats almost never.
The estimate therefore works from **floor area**: `footprint x floors x efficiency / m2 per person`.

**Flow:** 1 load · 2 data quality and scope · 3 distributions · 4 relationships and back-test · 5 information tiers and flags ·
6 baseline capacity · 7 export RAG material · 8 next steps.

There is no real capacity in OSM. *Baseline capacity* is a documented rule, not ground truth.

## 0. Configuration

In [ ]:
from pathlib import Path
import hashlib, json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (8, 4)

# --- data -------------------------------------------------------------------------------------------------
DATA_CSV = Path("data/housing_dataset.csv")
RAW_JSON = Path("data/overpass_raw.json")
DOWNLOAD_IF_MISSING = True
BBOX = (55.66, 12.55, 55.70, 12.62)   # south, west, north, east (central Copenhagen)
TILES = 3
LIMIT_PER_TILE = 3000
CHECK_COUNT = False                   # True: ask Overpass how many buildings the box holds
APARTMENT_TAGS = ["apartments", "residential"]   # OSM building values kept in the scope

# --- capacity formula (see section 6) ----------------------------------------------------------------------
M2_PER_PERSON = 40.0       # Copenhagen municipality, floor area per person (Danmarks Statistik Nyt nr. 164, 2016)
PERSONS_PER_FLAT = 1.8     # apartments, 1 Jan 2020 (Danmarks Statistik), used when the number of flats is known
LEGAL_M2_PER_PERSON = 20.0 # ceiling only (Lov om boligforhold para. 15 and the family reunification requirement)
FLOOR_EFFICIENCY = 0.80    # ASSUMPTION: share of footprint x floors that is dwelling area
ROOF_LEVEL_WEIGHT = 0.0    # ASSUMPTION: how much a roof level counts as a floor (0 = not at all)
M_PER_LEVEL = 3.0          # ASSUMPTION: storey height, turns a height into floors when levels are missing

# --- cleaning and flags -------------------------------------------------------------------------------------
MAX_LEVELS, MAX_HEIGHT_M, MAX_FLATS = 60, 200, 500
MIN_FOOTPRINT_M2, MAX_FOOTPRINT_M2 = 10, 20000
SMALL_FOOTPRINT_M2 = 60     # below this the building is probably an annex or shed: flagged
LARGE_FOOTPRINT_M2 = 3000   # above this it may be several buildings or not residential: flagged
FOOTPRINT_BINS = [0, 100, 250, 500, 1000, float("inf")]
FOOTPRINT_LABELS = ["under 100 m2", "100-250 m2", "250-500 m2", "500-1000 m2", "over 1000 m2"]

# --- RAG export ---------------------------------------------------------------------------------------------
MIN_N = 30                  # a derived figure needs at least this many buildings behind it
EVAL_SHARE = 0.30           # share of buildings kept out of the RAG examples, for testing
EXAMPLES_PER_TIER = 3
RANDOM_SEED = 42
OUT_DIR = Path("rag_export")

## 1. Load the data

If the CSV does not exist, the next cell downloads it tile by tile with `build_housing_dataset.py`. Finished tiles are saved in
`data/tiles/`, so a failure halfway only costs the missing tiles. To download from scratch, delete the CSV and `data/tiles/`.

In [ ]:
if not DATA_CSV.exists():
    if not DOWNLOAD_IF_MISSING:
        raise FileNotFoundError(f"{DATA_CSV} not found. Run build_housing_dataset.py first.")
    import importlib, build_housing_dataset as bhd
    importlib.reload(bhd)
    response = bhd.fetch_tiled(bhd.DEFAULT_URL, BBOX, LIMIT_PER_TILE, TILES, cache_dir="data/tiles")
    RAW_JSON.parent.mkdir(parents=True, exist_ok=True)
    RAW_JSON.write_text(json.dumps(response), encoding="utf-8")
    bhd.write_csv(bhd.rows_from_response(response), str(DATA_CSV))

all_buildings = pd.read_csv(DATA_CSV)
for col in ["levels", "flats", "height_m", "roof_levels", "footprint_m2"]:
    all_buildings[col] = pd.to_numeric(all_buildings[col], errors="coerce")
print("all residential building ways in the CSV:", len(all_buildings))
all_buildings.head()

### 1.1 Is the sample complete?

Each tile asks for at most `LIMIT_PER_TILE` buildings. If the number of rows is far below `TILES x TILES x LIMIT_PER_TILE`, no
tile was cut off. `CHECK_COUNT = True` also compares with the number of buildings Overpass reports for the box.

In [ ]:
capacity_of_download = LIMIT_PER_TILE * TILES * TILES
print("rows:", len(all_buildings), "| capacity of the download:", capacity_of_download)
if CHECK_COUNT:
    import importlib, build_housing_dataset as bhd
    importlib.reload(bhd)
    total = bhd.count_buildings(bhd.DEFAULT_URL, BBOX)
    print("building ways in the box:", total, "| share downloaded:", f"{len(all_buildings) / total:.0%}")
elif len(all_buildings) >= capacity_of_download:
    print("The download hit its limit: raise TILES or LIMIT_PER_TILE.")
else:
    print("No tile was full, so the sample is complete (assuming the CSV was made with these settings).")

## 2. Data quality and scope

### 2.1 Which building types are in the data, and what is in scope?

In [ ]:
types = all_buildings["building"].value_counts()
display(types.to_frame("buildings").assign(share_pct=(types / len(all_buildings) * 100).round(1),
                                            in_scope=lambda t: t.index.isin(APARTMENT_TAGS)))
df = all_buildings[all_buildings["building"].isin(APARTMENT_TAGS)].copy().reset_index(drop=True)
print(f"scope: {len(df)} apartment buildings ({len(df) / len(all_buildings):.1%} of the data); "
      f"left out: {len(all_buildings) - len(df)} (houses, terraced, dormitories)")

### 2.2 How often is each field filled? (apartment buildings)

This table decides what the request can ask for and which rules the data supports.

In [ ]:
FIELDS = ["footprint_m2", "levels", "roof_levels", "height_m", "flats", "name", "street_address"]
coverage = df[FIELDS].notna().mean().mul(100).round(1).rename("filled_pct")
display(coverage.to_frame())
coverage.plot.barh(title="Share of apartment buildings with the field filled (%)", xlim=(0, 100)).invert_yaxis()
plt.show()

### 2.3 Duplicates and implausible values

In [ ]:
print("duplicate (osm_type, osm_id):", df.duplicated(["osm_type", "osm_id"]).sum())
checks = {
    f"levels outside 1..{MAX_LEVELS}": ~df["levels"].between(1, MAX_LEVELS) & df["levels"].notna(),
    f"height_m outside 1..{MAX_HEIGHT_M}": ~df["height_m"].between(1, MAX_HEIGHT_M) & df["height_m"].notna(),
    f"flats outside 1..{MAX_FLATS}": ~df["flats"].between(1, MAX_FLATS) & df["flats"].notna(),
    f"footprint outside {MIN_FOOTPRINT_M2}..{MAX_FOOTPRINT_M2} m2": ~df["footprint_m2"].between(MIN_FOOTPRINT_M2, MAX_FOOTPRINT_M2) & df["footprint_m2"].notna(),
}
pd.Series({k: int(v.sum()) for k, v in checks.items()}, name="rows").to_frame()

Implausible values become missing in `*_clean` columns. The raw columns are kept.

In [ ]:
df["levels_clean"] = df["levels"].where(df["levels"].between(1, MAX_LEVELS))
df["height_clean"] = df["height_m"].where(df["height_m"].between(1, MAX_HEIGHT_M))
df["flats_clean"] = df["flats"].where(df["flats"].between(1, MAX_FLATS))
df["footprint_clean"] = df["footprint_m2"].where(df["footprint_m2"].between(MIN_FOOTPRINT_M2, MAX_FOOTPRINT_M2))
# a height can stand in for the number of levels
df["levels_eff"] = df["levels_clean"].fillna((df["height_clean"] / M_PER_LEVEL).round())
df.loc[df["levels_eff"] < 1, "levels_eff"] = np.nan
df[["levels_clean", "height_clean", "flats_clean", "footprint_clean"]].describe().round(1)

## 3. Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (col, title) in zip(axes.ravel(), [("levels_eff", "Levels"), ("footprint_clean", "Footprint (m2)"),
                                           ("flats_clean", "Flats"), ("height_clean", "Height (m)")]):
    data = df[col].dropna()
    if len(data):
        ax.hist(data, bins=30)
    ax.set_title(f"{title} (n={len(data)})")
plt.tight_layout(); plt.show()

## 4. Relationships and back-test

### 4.1 Height and levels

In [ ]:
both = df.dropna(subset=["height_clean", "levels_clean"]).copy()
both["m_per_level"] = both["height_clean"] / both["levels_clean"]
print("buildings with both:", len(both), "(a figure needs at least", MIN_N, ")")
if len(both):
    display(both["m_per_level"].describe().round(2).to_frame())

### 4.2 Back-test: the buildings that have a flat count

These are the only buildings where the real number of dwellings is known. For each, the table compares the
reference (`flats x persons per flat`) with the area formula used here and with the first formula (flats = usable area / 84.5 m2).
Three buildings prove nothing, and buildings with tagged flats may be unusual (for example newer or smaller flats), but the
direction of the error is a useful warning.

In [ ]:
flat_buildings = df.dropna(subset=["flats_clean", "levels_eff", "footprint_clean"]).copy()
flat_buildings["usable_area"] = flat_buildings["footprint_clean"] * flat_buildings["levels_eff"] * FLOOR_EFFICIENCY
flat_buildings["reference"] = (flat_buildings["flats_clean"] * PERSONS_PER_FLAT).round()
flat_buildings["area_formula"] = (flat_buildings["usable_area"] / M2_PER_PERSON).round()
flat_buildings["old_formula"] = (flat_buildings["usable_area"] / 84.5).round() * PERSONS_PER_FLAT
flat_buildings["usable_m2_per_flat"] = (flat_buildings["usable_area"] / flat_buildings["flats_clean"]).round(0)
flat_buildings["usable_m2_per_person_ref"] = (flat_buildings["usable_area"] / flat_buildings["reference"]).round(0)
flat_buildings["area_formula_error_pct"] = ((flat_buildings["area_formula"] / flat_buildings["reference"] - 1) * 100).round(0)
flat_buildings["old_formula_error_pct"] = ((flat_buildings["old_formula"] / flat_buildings["reference"] - 1) * 100).round(0)
print(f"{len(flat_buildings)} buildings with flats, levels and footprint")
flat_buildings[["osm_id", "levels_eff", "footprint_clean", "flats_clean", "usable_m2_per_flat", "usable_m2_per_person_ref",
                "reference", "area_formula", "area_formula_error_pct", "old_formula", "old_formula_error_pct"]]

### 4.3 Levels by footprint size

Buildings without a number of levels need an estimate. A bigger footprint usually means more levels, so the median number of
levels per footprint class is a better fill-in than one number for all buildings.

In [ ]:
df["fp_class"] = pd.cut(df["footprint_clean"], FOOTPRINT_BINS, labels=FOOTPRINT_LABELS, right=False)
known = df.dropna(subset=["levels_eff"])
levels_by_class = known.groupby("fp_class", observed=True)["levels_eff"].agg(median="median", buildings="count")
overall_levels = float(known["levels_eff"].median())
levels_by_class["used_for_imputation"] = np.where(levels_by_class["buildings"] >= MIN_N,
                                                   np.floor(levels_by_class["median"] + 0.5), np.floor(overall_levels + 0.5))
print("overall median levels:", overall_levels)
display(levels_by_class)
known.boxplot(column="levels_eff", by="fp_class", grid=False)
plt.suptitle(""); plt.title("Levels by footprint class"); plt.xticks(rotation=20); plt.show()

### 4.4 Roof levels

`roof:levels` counts attic or roof storeys. The table shows how it relates to `building:levels`.

In [ ]:
print("roof:levels filled:", int(df["roof_levels"].notna().sum()), "of", len(df))
display(pd.crosstab(df["levels_clean"].fillna(-1).astype(int).replace(-1, "missing"),
                    df["roof_levels"].fillna(-1).astype(int).replace(-1, "missing")))
print("buildings without levels that have roof levels:",
      int((df["levels_clean"].isna() & df["roof_levels"].notna()).sum()))

## 5. Information tiers and flags

The request will contain whatever the caller knows. The **tier** says which fields are known, and the RAG rules are written per tier.
Each tier has a **situation phrase** that is used in the rule chunks, the examples and the retrieval query, so retrieval matches on words
instead of numbers.

| Tier | Situation phrase | Confidence |
|---|---|---|
| A | flats known | high |
| B | levels known, flats unknown, footprint known | medium |
| C | levels known, flats unknown, footprint unknown | low |
| D | levels unknown, flats unknown, footprint known | low |
| E | levels unknown, flats unknown, footprint unknown | low |

**Flags** mark buildings that need care: a footprint under `SMALL_FOOTPRINT_M2` (probably an annex or shed) or over
`LARGE_FOOTPRINT_M2` (may be several buildings or not residential). Flagged buildings get confidence low and are kept out of the examples.

In [ ]:
def tier(row):
    if pd.notna(row["flats_clean"]):
        return "A"
    if pd.notna(row["levels_eff"]) and pd.notna(row["footprint_clean"]):
        return "B"
    if pd.notna(row["levels_eff"]):
        return "C"
    if pd.notna(row["footprint_clean"]):
        return "D"
    return "E"

def situation(row):
    if pd.notna(row["flats_clean"]):
        return "flats known"
    levels = "levels known" if pd.notna(row["levels_eff"]) else "levels unknown"
    footprint = "footprint known" if pd.notna(row["footprint_clean"]) else "footprint unknown"
    return f"{levels}, flats unknown, {footprint}"

df["tier"] = df.apply(tier, axis=1)
df["situation"] = df.apply(situation, axis=1)
df["flag"] = np.select([df["footprint_clean"] < SMALL_FOOTPRINT_M2, df["footprint_clean"] > LARGE_FOOTPRINT_M2],
                       ["small footprint", "large footprint"], default="")
display(df["tier"].value_counts().sort_index().to_frame("buildings").assign(share_pct=lambda t: (t["buildings"] / len(df) * 100).round(1)))
display(df["flag"].replace("", "none").value_counts().to_frame("buildings"))
df["tier"].value_counts().sort_index().plot.bar(title="Information tier (apartment buildings)")
plt.show()

## 6. Capacity baseline (a rule, not ground truth)

**Capacity = the typical number of residents.** For each building the baseline records the **reason** as text.

1. **Flats known (A):** `capacity = flats x PERSONS_PER_FLAT` (1.8, Danmarks Statistik, apartments, 2020).
2. **Otherwise:** `usable area = footprint x floors x FLOOR_EFFICIENCY`, then `capacity = usable area / M2_PER_PERSON` (40 m2, Copenhagen, 2016).
3. **Missing inputs:** no levels -> median levels of the footprint class (section 4.3); no footprint -> median footprint of all apartment buildings.
4. The capacity is at least 1 and at most `usable area / 20 m2` (legal ceiling).

Sources and legal context are in `housing-capacity-research.md`. `FLOOR_EFFICIENCY` and `ROOF_LEVEL_WEIGHT` are assumptions without a source.

In [ ]:
MEDIAN_FOOTPRINT = float(df["footprint_clean"].median())
class_levels = levels_by_class["used_for_imputation"].to_dict()
OVERALL_LEVELS = float(np.floor(overall_levels + 0.5))

def levels_for_imputation(footprint):
    if pd.isna(footprint):
        return OVERALL_LEVELS
    label = pd.cut([footprint], FOOTPRINT_BINS, labels=FOOTPRINT_LABELS, right=False)[0]
    return class_levels.get(label, OVERALL_LEVELS)

def baseline(row):
    """Returns (capacity, basis, confidence)."""
    if pd.notna(row["flats_clean"]):
        capacity = max(1, round(row["flats_clean"] * PERSONS_PER_FLAT))
        return capacity, f"{int(row['flats_clean'])} flats x {PERSONS_PER_FLAT:g} persons per flat", "high"
    notes = []
    footprint = row["footprint_clean"]
    if pd.isna(footprint):
        footprint = MEDIAN_FOOTPRINT
        notes.append(f"footprint missing, median footprint {footprint:.0f} m2 used")
    levels = row["levels_eff"]
    if pd.isna(levels):
        levels = levels_for_imputation(row["footprint_clean"])
        notes.append(f"levels missing, median of similar buildings ({levels:g}) used")
    roof = row["roof_levels"] if pd.notna(row["roof_levels"]) else 0
    floors = levels + ROOF_LEVEL_WEIGHT * roof
    usable = footprint * floors * FLOOR_EFFICIENCY
    capacity = usable / M2_PER_PERSON
    ceiling = max(1, int(usable // LEGAL_M2_PER_PERSON))
    final = max(1, min(round(capacity), ceiling))
    basis = (f"footprint {footprint:.0f} m2 x {floors:g} floors x {FLOOR_EFFICIENCY:g} = {usable:.0f} m2 usable; "
             f"{usable:.0f} / {M2_PER_PERSON:g} m2 per person = {capacity:.1f}")
    if final != max(1, round(capacity)):
        basis += f"; limited to {ceiling} by the 20 m2 per person ceiling"
    if notes:
        basis += "; " + "; ".join(notes)
    confidence = "medium" if row["tier"] == "B" else "low"
    if row["flag"]:
        confidence = "low"
        basis += f"; flag: {row['flag']}"
    return final, basis, confidence

result = df.apply(baseline, axis=1, result_type="expand")
df["baseline_capacity"], df["baseline_basis"], df["confidence"] = result[0], result[1], result[2]
display(df.groupby(["tier", "confidence"])["baseline_capacity"].agg(["count", "median", "min", "max"]))
df["baseline_capacity"].plot.hist(bins=50, title="Baseline capacity (apartment buildings)")
plt.show()
print("median capacity:", df["baseline_capacity"].median(), "| total:", int(df["baseline_capacity"].sum()))

### 6.1 Sensitivity

The unsourced and the uncertain parameters decide the result. The table shows the **total baseline capacity** for buildings without a
flat count, for different `M2_PER_PERSON` (rows) and `FLOOR_EFFICIENCY` (columns). The second table shows the effect of counting roof levels.
Use these ranges in the synopsis instead of one number.

In [ ]:
nf = df[df["flats_clean"].isna()].copy()
nf["fp"] = nf["footprint_clean"].fillna(MEDIAN_FOOTPRINT)
nf["lv"] = nf["levels_eff"].fillna(nf["footprint_clean"].map(levels_for_imputation))
nf["roof"] = nf["roof_levels"].fillna(0)

def total(m2pp, eff, weight=0.0):
    usable = nf["fp"] * (nf["lv"] + weight * nf["roof"]) * eff
    cap = np.maximum(1, np.minimum(np.rint(usable / m2pp), np.floor(usable / LEGAL_M2_PER_PERSON)))
    return int(cap.sum())

grid = pd.DataFrame({f"efficiency {e:g}": {f"{m:g} m2 per person": total(m, e) for m in [35, 40, 45, 54]} for e in [0.7, 0.8, 0.9]})
display(grid)
pd.DataFrame({"roof weight": [0, 0.5, 1.0], "total capacity": [total(M2_PER_PERSON, FLOOR_EFFICIENCY, w) for w in [0, 0.5, 1.0]]})

## 7. Export material for the RAG pipeline

The export is organised by **situation**, so a request that says "levels unknown, flats unknown, footprint known" retrieves the rule written for
that situation. Files in `rag_export/`:

| File | Content | Chunks (`## ` headings) |
|---|---|---|
| `apartment-rules.md` | one rule per tier, with formula, confidence and wording to use | one per tier |
| `apartment-reference.md` | what capacity means, Danmarks Statistik figures, legal ceiling, assumptions | one per topic |
| `apartment-profile.md` | what the data looks like, what is usually missing, levels by footprint class, limitations | one per topic |
| `apartment-examples.md` | worked examples per tier, unflagged buildings only | one per example |
| `eval_set.csv` | apartment buildings not used as examples, with an empty `capacity_label` column | - |
| `request_examples.json` | sample requests, one per tier | - |
| `README.md` | the same overview, plus the retrieval query format | - |

In [ ]:
OUT_DIR.mkdir(exist_ok=True)

def share(seed, osm_id):
    digest = hashlib.md5(f"{seed}-{osm_id}".encode()).hexdigest()
    return int(digest[:8], 16) / 16 ** 8

df["is_eval"] = df["osm_id"].map(lambda i: share(RANDOM_SEED, i) < EVAL_SHARE)
pool, evalset = df[~df["is_eval"]], df[df["is_eval"]]
print("example pool:", len(pool), "| held-out evaluation set:", len(evalset))

In [ ]:
n = len(df)
def pct(x): return f"{x:.0f}%"

def level_table():
    rows = ["| Footprint class | Median levels | Buildings with levels |", "|---|---|---|"]
    for label, r in levels_by_class.iterrows():
        used = f"{r['used_for_imputation']:g}"
        note = "" if r["buildings"] >= MIN_N else " (too few buildings, overall median used)"
        rows.append(f"| {label} | {used}{note} | {int(r['buildings'])} |")
    return rows

tier_counts = df["tier"].value_counts().sort_index()
profile = ["## Data profile: apartment buildings in central Copenhagen", "",
           f"Source: OpenStreetMap via the Overpass API, `building=apartments` and `building=residential`, {n} buildings in a box in central Copenhagen. "
           "OSM is volunteer-maintained and may be outdated.", "",
           "## What is usually missing", "",
           f"- Footprint area (computed from the building outline): available for {pct(df['footprint_clean'].notna().mean() * 100)} of the buildings.",
           f"- Number of levels (building:levels): available for {pct(df['levels_eff'].notna().mean() * 100)}; a height can replace it, but is given for only {pct(df['height_clean'].notna().mean() * 100)}.",
           f"- Number of flats (building:flats): available for {pct(df['flats_clean'].notna().mean() * 100)} ({int(df['flats_clean'].notna().sum())} buildings). It is almost always unknown, and a missing value is not zero.",
           "- Roof levels (roof:levels) are only given where levels are given, so they add no information when levels are missing.", "",
           "Typical buildings: median footprint " + f"{df['footprint_clean'].median():.0f} m2 (middle half {df['footprint_clean'].quantile(.25):.0f} to {df['footprint_clean'].quantile(.75):.0f} m2), "
           f"median {df['levels_eff'].median():.0f} levels (middle half {df['levels_eff'].quantile(.25):.0f} to {df['levels_eff'].quantile(.75):.0f}).", "",
           "Share of buildings per situation: " + ", ".join(f"tier {t} {pct(v / n * 100)}" for t, v in tier_counts.items()) + ".", "",
           "## Levels by footprint size (used when the number of levels is missing)", "",
           "A larger footprint usually means more levels. Median number of levels per footprint class in this data:", ""] + level_table() + [
           "", "## Limitations", "",
           "- There is no real capacity data. All estimates are reasoned from the building, not measured.",
           "- The data covers central Copenhagen only; other areas may differ.",
           f"- Buildings with a footprint under {SMALL_FOOTPRINT_M2} m2 are probably annexes or sheds, and over {LARGE_FOOTPRINT_M2} m2 may be several buildings or not residential. Use confidence low and say so.",
           "- A back-test on the few buildings with a known flat count found that the area formula underestimated the residents (see the notebook). Treat the result as a typical value, not a maximum.", ""]
(OUT_DIR / "apartment-profile.md").write_text("\n".join(profile), encoding="utf-8")

reference = ["## What capacity means", "",
             "Capacity is the typical number of residents of the apartment building when it is normally occupied. It is not the legal maximum, which would be about twice as high.", "",
             "## Reference figures from Danmarks Statistik", "",
             f"- Floor area per person in Copenhagen municipality: about {M2_PER_PERSON:g} m2 (1 January 2016, https://www.dst.dk/nyt/21543). The Danish average is 53.9 m2 (2026, https://www.dst.dk/da/Statistik/emner/borgere/boligforhold/personer-i-boliger).",
             f"- Persons per apartment (etagebolig): {PERSONS_PER_FLAT:g} (1 January 2020, https://www.dst.dk/nyt/30704).",
             "- Average apartment size in Copenhagen city: 84.5 m2 (2026, Statistikbanken BOL106, quoted by Bolius).", "",
             "## Legal ceiling", "",
             "- Rental housing: at most 2 persons per habitable room (Lov om boligforhold, paragraph 14). The number of rooms is not known here.",
             "- The municipality may allow more if there are at least 20 m2 per person (paragraph 15); the same figure is used in the family reunification housing requirement.",
             f"- The estimate is therefore limited to usable area / {LEGAL_M2_PER_PERSON:g} m2. This is a ceiling, never the estimate.", "",
             "## Assumptions without a source", "",
             f"- Dwelling area is {FLOOR_EFFICIENCY:g} of footprint x floors (walls, stairs and shops are not dwelling area).",
             f"- A roof level counts as {ROOF_LEVEL_WEIGHT:g} of a floor.",
             f"- A height is turned into floors with {M_PER_LEVEL:g} m per floor.", ""]
(OUT_DIR / "apartment-reference.md").write_text("\n".join(reference), encoding="utf-8")

mid = float(df["baseline_capacity"].median())
rules = ["# Rules for estimating the capacity of an apartment building", "",
         "Pick the rule whose heading matches what is known about the building. Answer with one JSON object: capacity (whole number, at least 1), confidence, reasoning.", "",
         "## Rule A: flats known", "",
         f"The number of flats is given. capacity = flats x {PERSONS_PER_FLAT:g} persons per flat, rounded. Confidence high. Levels and footprint can be ignored.", "",
         "## Rule B: levels known, flats unknown, footprint known", "",
         f"usable area = footprint x levels x {FLOOR_EFFICIENCY:g}. capacity = usable area / {M2_PER_PERSON:g} m2 per person, rounded, at least 1 and at most usable area / {LEGAL_M2_PER_PERSON:g}. Confidence medium. "
         "If a height is given instead of levels, levels = height / 3 m, rounded.", "",
         "## Rule C: levels known, flats unknown, footprint unknown", "",
         f"The footprint is missing. Use the median footprint of apartment buildings in this data, {MEDIAN_FOOTPRINT:.0f} m2, then apply rule B. Confidence low. State that the footprint was assumed.", "",
         "## Rule D: levels unknown, flats unknown, footprint known", "",
         f"The number of levels is missing. Take the median levels for the footprint class from the table in the data profile (for example {class_levels.get(FOOTPRINT_LABELS[2], OVERALL_LEVELS):g} levels for 250-500 m2), then apply rule B. Confidence low. State that the levels were assumed.", "",
         "## Rule E: levels unknown, flats unknown, footprint unknown", "",
         f"Almost nothing is known. Give the typical capacity of an apartment building in this data, about {mid:.0f} residents, with confidence low, and say that the building data is missing.", "",
         "## Rule F: flagged buildings (very small or very large footprint)", "",
         f"A footprint under {SMALL_FOOTPRINT_M2} m2 is probably an annex or shed, and over {LARGE_FOOTPRINT_M2} m2 may be several buildings or not residential. Apply the matching rule but use confidence low and mention the unusual size.", "",
         "## Answer format", "",
         'Answer with one JSON object and nothing else: {"capacity": 54, "confidence": "medium", "reasoning": "footprint 630 m2 x 5 levels x 0.8 = 2520 m2 usable, / 40 m2 per person = 63"}. '
         "Confidence is high, medium or low as stated in the rule. The reasoning names the numbers used and every value that was assumed.", ""]
(OUT_DIR / "apartment-rules.md").write_text("\n".join(rules), encoding="utf-8")
print("rules, reference and profile written")
print("\n".join(rules))

In [ ]:
def fmt(value, integer=False):
    if pd.isna(value):
        return "unknown"
    return str(int(round(value))) if integer else f"{value:g}"

def context_lines(r):
    return [f"buildingType: apartment building",
            f"footprintM2: {fmt(r['footprint_clean'], True)}",
            f"numberOfLevels: {fmt(r['levels_eff'], True)}",
            f"numberOfFlats: {fmt(r['flats_clean'], True)}",
            f"heightMeters: {fmt(r['height_clean'])}"]

candidates = pool[pool["flag"] == ""]
parts = []
for t, part in candidates.groupby("tier"):
    parts.append(part.sample(min(len(part), EXAMPLES_PER_TIER), random_state=RANDOM_SEED))
examples = pd.concat(parts).sort_values("tier") if parts else candidates.iloc[0:0]

blocks = ["# Worked examples: apartment buildings with a reasoned capacity", "",
          "The capacities come from the rules in this material, not from measurements. They show how to reason.", ""]
for i, (_, r) in enumerate(examples.iterrows(), 1):
    blocks += [f"## Example {i}: {r['situation']}"] + context_lines(r) + [
        f"Reasoning: {r['baseline_basis']}.", f"capacity: {int(r['baseline_capacity'])}", f"confidence: {r['confidence']}", ""]
(OUT_DIR / "apartment-examples.md").write_text("\n".join(blocks), encoding="utf-8")

eval_cols = ["osm_type", "osm_id", "situation", "tier", "flag", "footprint_clean", "levels_clean", "levels_eff", "flats_clean",
             "height_clean", "baseline_capacity", "confidence", "baseline_basis"]
evalset[eval_cols].assign(capacity_label="").to_csv(OUT_DIR / "eval_set.csv", index=False)

samples = []
for t, part in evalset.groupby("tier"):
    r = part.iloc[0]
    samples.append({"id": f"{r['osm_type']}/{int(r['osm_id'])}", "tier": t, "situation": r["situation"],
                    "request": {"requestType": "HOUSING_CAPACITY", "useRag": True, "context": "\n".join(context_lines(r))},
                    "baseline_capacity": int(r["baseline_capacity"]), "baseline_basis": r["baseline_basis"]})
(OUT_DIR / "request_examples.json").write_text(json.dumps(samples, indent=2, ensure_ascii=False), encoding="utf-8")

readme = f"""# RAG material for apartment capacity

Generated by `housing_eda.ipynb` from {n} apartment buildings in central Copenhagen (OpenStreetMap).

| File | Use |
|---|---|
| apartment-rules.md | knowledge source: one rule per situation (tier) |
| apartment-reference.md | knowledge source: meaning of capacity, statistics, legal ceiling, assumptions |
| apartment-profile.md | knowledge source: data profile, missing data, levels by footprint class, limitations |
| apartment-examples.md | knowledge source: worked examples per situation |
| eval_set.csv | buildings not in the examples; fill `capacity_label` by hand for testing |
| request_examples.json | sample requests in the proposed request format |
| baseline-parameters.properties | numbers of the baseline rule; copy to the LLM API resources (not a knowledge source) |

Chunking: every `## ` heading starts a new chunk (the LLM API already splits this way).

Retrieval query: describe the request with the same words as the headings, for example
`apartment building, levels unknown, flats unknown, footprint known`. The situation phrase is one of:
flats known / levels known, flats unknown, footprint known / levels known, flats unknown, footprint unknown /
levels unknown, flats unknown, footprint known / levels unknown, flats unknown, footprint unknown.
Embeddings handle words better than numbers, so the query should not rely on the numbers.
"""
(OUT_DIR / "README.md").write_text(readme, encoding="utf-8")

# The numbers of the baseline rule, read by the LLM API (BaselineParameters) so both use the same figures.
finite_bins = [b for b in FOOTPRINT_BINS if np.isfinite(b)]
by_class = [float(class_levels[label]) for label in FOOTPRINT_LABELS]
param_lines = ["# Written by housing_eda.ipynb. Read by the LLM API.",
               f"m2PerPerson={M2_PER_PERSON:g}", f"personsPerFlat={PERSONS_PER_FLAT:g}", f"legalM2PerPerson={LEGAL_M2_PER_PERSON:g}",
               f"floorEfficiency={FLOOR_EFFICIENCY:g}", f"mPerLevel={M_PER_LEVEL:g}", f"medianFootprintM2={MEDIAN_FOOTPRINT:.1f}",
               f"overallLevels={OVERALL_LEVELS:g}", f"smallFootprintM2={SMALL_FOOTPRINT_M2:g}", f"largeFootprintM2={LARGE_FOOTPRINT_M2:g}",
               "footprintBins=" + ",".join(f"{b:g}" for b in finite_bins),
               "levelsByClass=" + ",".join(f"{v:g}" for v in by_class)]
(OUT_DIR / "baseline-parameters.properties").write_text("\n".join(param_lines) + "\n", encoding="utf-8")
print("\n".join(param_lines))
print(len(examples), "examples,", len(evalset), "evaluation rows,", len(samples), "sample requests")
print("\n".join(blocks[:30]))

## 8. What to check after a run

- Section 1.1: is the sample complete?
- Section 4.2: how far off is the area formula on buildings with a known flat count? (three buildings in the last run, all underestimated)
- Section 5: how many buildings are flagged, and how many are tier D (levels imputed)?
- Section 6.1: how wide is the range of total capacity? Use the range in the synopsis.
- Read `apartment-rules.md` and the examples: would a person understand each rule and example without the notebook?
- Hand-label 40-60 rows of `eval_set.csv`, or find an official source for the number of dwellings, before comparing the LLM with the baseline.